# 03 — Synthetic Data Generation

**DRISHTI** — Synthetic debris compositor for rare-class augmentation

Generates fishing nets, pipes, and cylinders pasted onto real seabed backgrounds
with acoustic shadow simulation and auto-generated YOLO-seg polygon labels.

**Run on:** Any environment (CPU only, no GPU needed).

In [ ]:
!pip install -q opencv-python-headless numpy matplotlib

In [ ]:
import numpy as np
import cv2
import random
import math
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
random.seed(42)
np.random.seed(42)
print('Setup complete.')

## 1. Procedural Debris Generators

Each generator creates an (object_image, alpha_mask) pair simulating how the
debris would appear in a forward-looking sonar (FLS) scan.

In [ ]:
def generate_net(size=(120, 120), mesh_spacing=12, line_thickness=2):
    h, w = size
    obj = np.zeros((h, w), dtype=np.uint8)
    mask = np.zeros((h, w), dtype=np.uint8)
    
    for y in range(0, h, mesh_spacing):
        pts = [[x, min(max(y + int(random.gauss(0, mesh_spacing*0.1)), 0), h-1)] for x in range(0, w, 4)]
        if len(pts) >= 2:
            cv2.polylines(obj, [np.array(pts, np.int32)], False, 200, line_thickness)
    for x in range(0, w, mesh_spacing):
        pts = [[min(max(x + int(random.gauss(0, mesh_spacing*0.1)), 0), w-1), y] for y in range(0, h, 4)]
        if len(pts) >= 2:
            cv2.polylines(obj, [np.array(pts, np.int32)], False, 200, line_thickness)
    
    noise = np.random.normal(0, 15, (h, w)).astype(np.float32)
    obj = np.clip(obj.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    _, mask = cv2.threshold(obj, 30, 255, cv2.THRESH_BINARY)
    mask = cv2.dilate(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3)), iterations=1)
    return obj, mask

def generate_pipe(size=(60, 180), orientation=0.0):
    h, w = size
    obj = np.zeros((h, w), dtype=np.uint8)
    mask = np.zeros((h, w), dtype=np.uint8)
    cy, cx = h//2, w//2
    pipe_h, pipe_w = int(h*0.4), int(w*0.85)
    top, bot = cy - pipe_h//2, cy + pipe_h//2
    left, right = cx - pipe_w//2, cx + pipe_w//2
    for row in range(max(top,0), min(bot,h)):
        prog = (row - top) / max(pipe_h, 1)
        intensity = int(220 * math.exp(-((prog - 0.3)**2) / 0.05))
        intensity = max(min(intensity, 255), 40)
        obj[row, max(left,0):min(right,w)] = intensity
        mask[row, max(left,0):min(right,w)] = 255
    if abs(orientation) > 1:
        M = cv2.getRotationMatrix2D((cx, cy), orientation, 1.0)
        obj = cv2.warpAffine(obj, M, (w, h))
        mask = cv2.warpAffine(mask, M, (w, h))
    return obj, mask

def generate_cylinder(size=(80, 80)):
    h, w = size
    obj = np.zeros((h, w), dtype=np.uint8)
    mask = np.zeros((h, w), dtype=np.uint8)
    cy, cx, r = h//2, w//2, min(h,w)//3
    cv2.circle(obj, (cx,cy), r, 200, -1)
    cv2.circle(mask, (cx,cy), r, 255, -1)
    cv2.circle(obj, (cx,cy), int(r*0.7), 140, -1)
    return obj, mask

print('Generators ready.')

In [ ]:
# ---- Visualise individual generators ----
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col, (name, gen_fn, args) in enumerate([
    ('Fishing Net', generate_net, {}),
    ('Pipe', generate_pipe, {'orientation': 15}),
    ('Cylinder', generate_cylinder, {}),
]):
    obj, mask = gen_fn(**args)
    axes[0, col].imshow(obj, cmap='gray')
    axes[0, col].set_title(f'{name} — Object', fontsize=11)
    axes[0, col].axis('off')
    axes[1, col].imshow(mask, cmap='gray')
    axes[1, col].set_title(f'{name} — Mask', fontsize=11)
    axes[1, col].axis('off')

plt.suptitle('Procedural Debris Generators', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Procedural Seabed Backgrounds

In [ ]:
def generate_seabed(size=640):
    base = np.zeros((size, size), dtype=np.float64)
    for scale in [4, 8, 16, 32, 64]:
        small = np.random.randn(scale, scale)
        up = cv2.resize(small, (size, size), interpolation=cv2.INTER_CUBIC)
        base += up * (64.0 / scale)
    base = (base - base.min()) / (base.max() - base.min() + 1e-10) * 180 + 20
    base += np.random.exponential(1.0, (size, size)) * 15
    return np.clip(base, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i in range(4):
    bg = generate_seabed()
    axes[i].imshow(bg, cmap='gray')
    axes[i].set_title(f'Seabed #{i+1}')
    axes[i].axis('off')
plt.suptitle('Procedural Seabed Textures', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Acoustic Shadow Simulation

In [ ]:
def add_shadow(bg, mask, paste_y, paste_x, length_factor=1.5):
    result = bg.copy()
    h, w = mask.shape
    shadow_len = int(h * length_factor)
    for dy in range(1, shadow_len + 1):
        sy = paste_y + dy
        if sy >= result.shape[0]:
            break
        for dx in range(w):
            sx = paste_x + dx
            if 0 <= sx < result.shape[1] and mask[min(dy, h-1), dx] > 0:
                falloff = math.exp(-dy / max(shadow_len * 0.5, 1))
                darkness = 0.15 + 0.35 * (1 - falloff)
                result[sy, sx] = int(result[sy, sx] * darkness)
    return result

# Demo
bg = generate_seabed()
net_obj, net_mask = generate_net((100, 100))
py, px = 200, 250

# Paste
composite = bg.copy()
oh, ow = net_obj.shape
roi = composite[py:py+oh, px:px+ow]
mask_norm = net_mask / 255.0 * 0.8
blended = roi * (1 - mask_norm) + net_obj * mask_norm
composite[py:py+oh, px:px+ow] = np.clip(blended, 0, 255).astype(np.uint8)

# Shadow
with_shadow = add_shadow(composite, net_mask, py + oh, px, 1.5)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(bg, cmap='gray'); axes[0].set_title('Background'); axes[0].axis('off')
axes[1].imshow(composite, cmap='gray'); axes[1].set_title('+ Net Composited'); axes[1].axis('off')
axes[2].imshow(with_shadow, cmap='gray'); axes[2].set_title('+ Acoustic Shadow'); axes[2].axis('off')
plt.suptitle('Composite Pipeline Demo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Batch Generation with Auto-Labels

In [ ]:
def mask_to_yolo_polygon(mask, class_id, img_w, img_h, min_area=50):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    cnt = max(contours, key=cv2.contourArea)
    if cv2.contourArea(cnt) < min_area:
        return None
    eps = 0.01 * cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, eps, True)
    if len(approx) < 3:
        return None
    pts = approx.squeeze()
    coords = []
    for x, y in pts:
        coords.extend([f'{x/img_w:.6f}', f'{y/img_h:.6f}'])
    return f'{class_id} ' + ' '.join(coords)

# Generate a batch
OUT_DIR = Path('synthetic_demo')
(OUT_DIR / 'images').mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'labels').mkdir(parents=True, exist_ok=True)

N_IMAGES = 50
IMG_SIZE = 640
generators = [(11, generate_net), (11, generate_pipe), (11, generate_cylinder)]

for idx in range(N_IMAGES):
    bg = generate_seabed(IMG_SIZE)
    labels = []
    
    for _ in range(random.randint(1, 3)):
        cls_id, gen = random.choice(generators)
        scale = random.uniform(0.06, 0.25)
        obj_size = max(int(IMG_SIZE * scale), 20)
        
        if gen == generate_pipe:
            obj, msk = gen((obj_size, obj_size*3), random.uniform(-30,30))
        else:
            obj, msk = gen((obj_size, obj_size))
        
        oh, ow = obj.shape
        max_y, max_x = IMG_SIZE - oh, IMG_SIZE - ow
        if max_y <= 0 or max_x <= 0:
            continue
        py, px = random.randint(0, max_y), random.randint(0, max_x)
        opacity = random.uniform(0.5, 1.0)
        
        roi = bg[py:py+oh, px:px+ow]
        mn = (msk / 255.0) * opacity
        bg[py:py+oh, px:px+ow] = np.clip(roi*(1-mn) + obj*mn, 0, 255).astype(np.uint8)
        bg = add_shadow(bg, msk, py+oh, px, random.uniform(0.8, 2.0))
        
        full_mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
        full_mask[py:py+oh, px:px+ow] = msk
        line = mask_to_yolo_polygon(full_mask, cls_id, IMG_SIZE, IMG_SIZE)
        if line:
            labels.append(line)
    
    cv2.imwrite(str(OUT_DIR / 'images' / f'syn_{idx:04d}.png'), bg)
    with open(OUT_DIR / 'labels' / f'syn_{idx:04d}.txt', 'w') as f:
        f.write('\n'.join(labels) + ('\n' if labels else ''))

print(f'Generated {N_IMAGES} synthetic images → {OUT_DIR}')

In [ ]:
# ---- Quality check: show a few results with overlaid labels ----
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i in range(8):
    img = cv2.imread(str(OUT_DIR / 'images' / f'syn_{i:04d}.png'), cv2.IMREAD_GRAYSCALE)
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    
    lbl_path = OUT_DIR / 'labels' / f'syn_{i:04d}.txt'
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 7:
                    continue
                coords = list(map(float, parts[1:]))
                pts = np.array([[int(coords[j]*IMG_SIZE), int(coords[j+1]*IMG_SIZE)]
                                for j in range(0, len(coords)-1, 2)], np.int32)
                cv2.polylines(vis, [pts], True, (0, 255, 0), 2)
    
    axes[i].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f'syn_{i:04d}', fontsize=9)
    axes[i].axis('off')

plt.suptitle('Synthetic Composites with Auto-Generated Labels (green polygons)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Summary

- Generated procedural net/pipe/cylinder debris silhouettes
- Pasted onto procedural seabed backgrounds with acoustic shadow simulation
- Auto-generated YOLO-seg polygon labels from pasted masks

For full-scale generation (500+ images), use `ml/scripts/build_synthetic_data.py`.

**Next:** `04_confidence_calibration.ipynb`